In [4]:
from pathlib import Path

for p in Path("checkpoints").rglob("*"):
    if p.is_file():
        print(p)

checkpoints\efficientnet_b0_efficientnetb0_checkpoint.pth\efficientnet_b0_efficientnetb0_checkpoint\byteorder
checkpoints\efficientnet_b0_efficientnetb0_checkpoint.pth\efficientnet_b0_efficientnetb0_checkpoint\data.pkl
checkpoints\efficientnet_b0_efficientnetb0_checkpoint.pth\efficientnet_b0_efficientnetb0_checkpoint\version
checkpoints\efficientnet_b0_efficientnetb0_checkpoint.pth\efficientnet_b0_efficientnetb0_checkpoint\.data\serialization_id
checkpoints\efficientnet_b0_efficientnetb0_checkpoint.pth\efficientnet_b0_efficientnetb0_checkpoint\data\0
checkpoints\efficientnet_b0_efficientnetb0_checkpoint.pth\efficientnet_b0_efficientnetb0_checkpoint\data\1
checkpoints\efficientnet_b0_efficientnetb0_checkpoint.pth\efficientnet_b0_efficientnetb0_checkpoint\data\10
checkpoints\efficientnet_b0_efficientnetb0_checkpoint.pth\efficientnet_b0_efficientnetb0_checkpoint\data\100
checkpoints\efficientnet_b0_efficientnetb0_checkpoint.pth\efficientnet_b0_efficientnetb0_checkpoint\data\101
checkpoint

In [15]:
import torch
import torch.nn as nn
import torchvision.models as models

checkpoint_path = Path(r"C:\CropDisease\AI_Based_Crop_Disease_Detection\backend\checkpoints\efficientnet_b0_efficientnetb0_checkpoint .pth")

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys())[:20])

Checkpoint type: <class 'dict'>
Checkpoint keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'best_val_acc']


In [16]:
import torch
import torch.nn as nn
import torchvision.models as models

CLASS_NAMES = [
    "Corn_Blight", "Corn_Common_Rust", "Corn_Gray_Leaf_Spot", "Corn_Healthy",
    "Cotton_Aphids", "Cotton_Army worm", "Cotton_Bacterial blight", "Cotton_Healthy",
    "Jute_Cescospora Leaf Spot", "Jute_Golden Mosaic", "Jute_Healthy Leaf",
    "Rice_Bacterialblight", "Rice_Blast", "Rice_Brownspot", "Rice_Tungro",
    "Sugarcane_Healthy", "Sugarcane_Mosaic", "Sugarcane_RedRot", "Sugarcane_Rust",
    "Sugarcane_Yellow", "Wheat_Healthy", "Wheat_septoria", "Wheat_stripe_rust",
]

model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    len(CLASS_NAMES)
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded successfully!")
print("Best validation accuracy:", checkpoint.get("best_val_acc"))

Model loaded successfully!
Best validation accuracy: 0.9687630154102457


In [17]:
from PIL import Image
from torchvision import transforms

image_path = r"img1.jpeg"

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

image = Image.open(image_path).convert("RGB")
tensor = preprocess(image).unsqueeze(0)

with torch.inference_mode():
    logits = model(tensor)
    probabilities = torch.softmax(logits, dim=1)[0]
    top_probs, top_indices = torch.topk(probabilities, 3)

for prob, idx in zip(top_probs, top_indices):
    print(f"{CLASS_NAMES[idx.item()]}: {prob.item() * 100:.2f}%")

Corn_Blight: 99.53%
Corn_Gray_Leaf_Spot: 0.46%
Corn_Common_Rust: 0.00%


In [22]:

import sqlite3
from pathlib import Path

DB_PATH = Path.cwd()/ "crop_disease.db"

SCHEMA = """
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS crops (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS diseases (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    crop_id INTEGER NOT NULL,
    name TEXT NOT NULL,
    description TEXT,
    is_healthy BOOLEAN NOT NULL DEFAULT 0,
    FOREIGN KEY (crop_id) REFERENCES crops(id),
    UNIQUE (crop_id, name)
);

CREATE TABLE IF NOT EXISTS remediations (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    disease_id INTEGER NOT NULL,
    type TEXT NOT NULL CHECK (type IN ('home', 'chemical')),
    description TEXT NOT NULL,
    precautions TEXT,
    FOREIGN KEY (disease_id) REFERENCES diseases(id)
);

CREATE TABLE IF NOT EXISTS sources (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    url TEXT NOT NULL UNIQUE,
    publisher TEXT
);

CREATE TABLE IF NOT EXISTS remediation_sources (
    remediation_id INTEGER NOT NULL,
    source_id INTEGER NOT NULL,
    PRIMARY KEY (remediation_id, source_id),
    FOREIGN KEY (remediation_id)
        REFERENCES remediations(id) ON DELETE CASCADE,
    FOREIGN KEY (source_id)
        REFERENCES sources(id) ON DELETE CASCADE
);
"""

def init_db():
    with sqlite3.connect(DB_PATH) as conn:
        conn.executescript(SCHEMA)

    print(f"Database initialized at: {DB_PATH}")


if __name__ == "__main__":
    init_db()


Database initialized at: c:\CropDisease\AI_Based_Crop_Disease_Detection\backend\crop_disease.db


In [23]:
conn = sqlite3.connect(DB_PATH)
commands = [
    "ALTER TABLE crops ADD COLUMN name_mr TEXT;",
    "ALTER TABLE diseases ADD COLUMN name_mr TEXT;",
    "ALTER TABLE diseases ADD COLUMN description_mr TEXT;",
    "ALTER TABLE remediations ADD COLUMN description_mr TEXT;",
    "ALTER TABLE remediations ADD COLUMN precautions_mr TEXT;"
]

for command in commands:
    conn.execute(command)

conn.commit()

In [24]:

# First batch: 10 model classes, their remediations and sources.
# Uses your existing SQLite connection: conn

crops_data = [
    ("Rice", "तांदूळ"),
    ("Corn", "मका"),
    ("Cotton", "कापूस"),
    ("Wheat", "गहू"),
    ("Sugarcane", "ऊस"),
]

for name, name_mr in crops_data:
    conn.execute(
        "INSERT OR IGNORE INTO crops (name, name_mr) VALUES (?, ?)",
        (name, name_mr)
    )

# (crop, model_class, Marathi name, English description, Marathi description)
diseases_data = [
    ("Rice", "Rice_Bacterialblight", "भातावरील जिवाणूजन्य करपा",
     "Bacterial leaf blight. Yellowing, water-soaked leaf streaks may occur.",
     "पानांवर पाणथळ पट्टे आणि पिवळेपणा दिसू शकतो."),
    ("Rice", "Rice_Blast", "भातावरील करपा (ब्लास्ट)",
     "Fungal disease that can cause spindle-shaped leaf lesions and neck infection.",
     "पानांवर लांबट डाग आणि कणसाच्या मानेजवळ संसर्ग होऊ शकतो."),
    ("Rice", "Rice_Brownspot", "भातावरील तपकिरी ठिपके",
     "Brown spot disease, producing brown round or oval lesions on leaves.",
     "पानांवर गोल किंवा अंडाकृती तपकिरी ठिपके दिसतात."),
    ("Rice", "Rice_Tungro", "भातावरील टुंग्रो रोग",
     "Viral disease associated with stunting, reduced tillering and yellow-orange leaves.",
     "झाडांची वाढ खुंटणे, फुटवे कमी होणे आणि पाने पिवळी-नारिंगी होणे अशी लक्षणे दिसू शकतात."),
    ("Corn", "Corn_Blight", "मक्यावरील पानांचा करपा",
     "Maize leaf blight can cause elongated grey-green or tan leaf lesions.",
     "पानांवर लांबट करड्या-हिरव्या किंवा तपकिरी डाग दिसू शकतात."),
    ("Corn", "Corn_Common_Rust", "मक्यावरील तांबेरा",
     "Common rust produces cinnamon-brown powdery pustules on maize leaves.",
     "मक्याच्या पानांवर तांबूस-तपकिरी भुकटीसारखे फोड दिसतात."),
    ("Cotton", "Cotton_Aphids", "कापसावरील मावा",
     "Aphids feed on tender shoots and leaf undersides, causing curling and stunted growth.",
     "मावा कोवळ्या फांद्यांवर व पानांच्या खालच्या बाजूस आढळतो; पाने कुरळे होऊ शकतात."),
    ("Cotton", "Cotton_Bacterial blight", "कापसावरील जिवाणूजन्य करपा",
     "Bacterial blight may cause angular leaf spots, vein blackening and boll symptoms.",
     "पानांवर कोनीय डाग, शिरा काळ्या पडणे आणि बोंडांवर लक्षणे दिसू शकतात."),
    ("Wheat", "Wheat_stripe_rust", "गव्हावरील पिवळा तांबेरा",
     "Stripe rust produces yellow to orange-yellow pustules arranged in stripes on leaves.",
     "गव्हाच्या पानांवर पिवळ्या ते नारिंगी-पिवळ्या फोडांच्या पट्ट्या दिसतात."),
    ("Sugarcane", "Sugarcane_RedRot", "उसाचा लाल कुज रोग",
     "Red rot is a sugarcane disease that can affect stalks and internal tissue.",
     "उसाच्या खोडाला आणि आतील ऊतींना लाल कुज रोगाचा प्रादुर्भाव होऊ शकतो."),
]

for crop, name, name_mr, description, description_mr in diseases_data:
    crop_id = conn.execute(
        "SELECT id FROM crops WHERE name = ?", (crop,)
    ).fetchone()[0]

    conn.execute("""
        INSERT OR IGNORE INTO diseases
            (crop_id, name, description, is_healthy, name_mr, description_mr)
        VALUES (?, ?, ?, 0, ?, ?)
    """, (crop_id, name, description, name_mr, description_mr))

# Official/technical sources used for this batch
sources_data = [
    ("TNAU Rice Disease Management",
     "https://www.agritech.tnau.ac.in/org_farm/orgfarm_agridiseases.html",
     "Tamil Nadu Agricultural University"),
    ("TNAU Maize Leaf Blight",
     "https://agritech.tnau.ac.in/crop_protection/maize_disease_new/maize_2.html",
     "Tamil Nadu Agricultural University"),
    ("TNAU Maize Common Rust",
     "https://agritech.tnau.ac.in/crop_protection/maize_disease_new/maize_4.html",
     "Tamil Nadu Agricultural University"),
    ("TNAU Cotton Aphid Management",
     "https://agritech.tnau.ac.in/crop_protection/cotton/crop_prot_crop_insectpest%20_cotton_11.html",
     "Tamil Nadu Agricultural University"),
    ("TNAU Cotton Bacterial Blight",
     "https://agritech.tnau.ac.in/crop_protection/cotton_diseases/cotton_d9.html",
     "Tamil Nadu Agricultural University"),
    ("TNAU Wheat Stripe Rust",
     "https://agritech.tnau.ac.in/crop_protection/crop_prot_crop%20diseases_cereals_wheat_3.html",
     "Tamil Nadu Agricultural University"),
    ("TNAU Sugarcane Red Rot",
     "https://agritech.tnau.ac.in/govt_schemes_services/aas/sugarcane.html",
     "Tamil Nadu Agricultural University"),
]

for title, url, publisher in sources_data:
    conn.execute("""
        INSERT OR IGNORE INTO sources (title, url, publisher)
        VALUES (?, ?, ?)
    """, (title, url, publisher))

# (model class, type, English remediation, Marathi remediation, source title)
remediations_data = [
    ("Rice_Bacterialblight", "home",
     "Use healthy seed, avoid excess nitrogen, and remove infected crop debris.",
     "निरोगी बियाणे वापरा, नत्राचा अतिरेक टाळा आणि रोगट अवशेष काढून नष्ट करा.",
     "TNAU Rice Disease Management"),
    ("Rice_Blast", "home",
     "Use a locally recommended tolerant variety and manage crop residues.",
     "स्थानिक शिफारशीनुसार सहनशील वाण वापरा आणि पिकांचे अवशेष योग्य प्रकारे व्यवस्थापित करा.",
     "TNAU Rice Disease Management"),
    ("Rice_Brownspot", "home",
     "Use healthy seed and follow locally recommended crop nutrition and residue management.",
     "निरोगी बियाणे वापरा आणि स्थानिक शिफारशीनुसार खत व्यवस्थापन व अवशेष व्यवस्थापन करा.",
     "TNAU Rice Disease Management"),
    ("Rice_Tungro", "home",
     "Use tolerant varieties where recommended and manage weed hosts and leafhopper vectors.",
     "शिफारस केलेले सहनशील वाण वापरा आणि तणांवरील आश्रयदाते व रोगवाहक तुडतुडे यांचे व्यवस्थापन करा.",
     "TNAU Rice Disease Management"),
    ("Corn_Blight", "home",
     "Remove or properly bury infected maize crop residues to reduce disease carryover.",
     "रोगाचा प्रसार कमी करण्यासाठी संक्रमित मक्याचे अवशेष काढून नष्ट करा किंवा योग्यरीत्या जमिनीत गाडा.",
     "TNAU Maize Leaf Blight"),
    ("Corn_Common_Rust", "home",
     "Remove infected crop residues and seek advice on locally suitable resistant varieties.",
     "संक्रमित अवशेष काढून टाका आणि स्थानिक परिस्थितीसाठी योग्य प्रतिकारक्षम वाणांची माहिती घ्या.",
     "TNAU Maize Common Rust"),
    ("Cotton_Aphids", "home",
     "Monitor aphid populations and conserve beneficial insects; avoid unnecessary spraying.",
     "माव्याच्या प्रादुर्भावाचे निरीक्षण करा, उपयुक्त कीटकांचे संरक्षण करा आणि अनावश्यक फवारणी टाळा.",
     "TNAU Cotton Aphid Management"),
    ("Cotton_Bacterial blight", "home",
     "Remove infected plant debris and use healthy seed; follow crop rotation where appropriate.",
     "संक्रमित अवशेष नष्ट करा, निरोगी बियाणे वापरा आणि योग्य ठिकाणी पीक फेरपालट करा.",
     "TNAU Cotton Bacterial Blight"),
    ("Wheat_stripe_rust", "home",
     "Use locally recommended resistant varieties and avoid excessive nitrogen fertilization.",
     "स्थानिक शिफारशीनुसार प्रतिकारक्षम वाण वापरा आणि नत्र खतांचा अतिरेक टाळा.",
     "TNAU Wheat Stripe Rust"),
    ("Sugarcane_RedRot", "home",
     "Use healthy planting setts, remove affected clumps, and follow recommended crop rotation.",
     "निरोगी लागवड बेणे वापरा, रोगट बेट काढून टाका आणि शिफारस केलेले पीक फेरपालट करा.",
     "TNAU Sugarcane Red Rot"),
]

# Insert remediations and connect each one to its source
for model_class, rem_type, description, description_mr, source_title in remediations_data:
    disease_id = conn.execute(
        "SELECT id FROM diseases WHERE name = ?", (model_class,)
    ).fetchone()[0]

    conn.execute("""
        INSERT OR IGNORE INTO remediations
            (disease_id, type, description, description_mr)
        VALUES (?, ?, ?, ?)
    """, (disease_id, rem_type, description, description_mr))

    remediation_id = conn.execute("""
        SELECT id FROM remediations
        WHERE disease_id = ? AND type = ? AND description = ?
    """, (disease_id, rem_type, description)).fetchone()[0]

    source_id = conn.execute(
        "SELECT id FROM sources WHERE title = ?", (source_title,)
    ).fetchone()[0]

    conn.execute("""
        INSERT OR IGNORE INTO remediation_sources
            (remediation_id, source_id)
        VALUES (?, ?)
    """, (remediation_id, source_id))

conn.commit()
print("Inserted/checked 10 disease classes and linked remediation sources.")


Inserted/checked 10 disease classes and linked remediation sources.


In [37]:

chemical_sources = [
    (
        "TNAU Rice Blast Chemical Management",
        "https://agritech.tnau.ac.in/expert_system/paddy/cpdisblast.html",
        "Tamil Nadu Agricultural University"
    ),
    (
        "TNAU Wheat Stripe Rust Management",
        "https://agritech.tnau.ac.in/crop_protection/crop_prot_crop%20diseases_cereals_wheat_3.html",
        "Tamil Nadu Agricultural University"
    ),
    (
        "TNAU Cotton Aphid Integrated Management",
        "https://agritech.tnau.ac.in/crop_protection/cotton/crop_prot_crop_insectpest%20_cotton_11.html",
        "Tamil Nadu Agricultural University"
    ),
    (
        "TNAU Crop Production Guide 2020 - Maize Rust",
        "https://tnau.ac.in/site/research/wp-content/uploads/sites/60/2020/02/Agriculture-CPG-2020.pdf",
        "Tamil Nadu Agricultural University"
    ),
]

for title, url, publisher in chemical_sources:
    conn.execute("""
        INSERT OR IGNORE INTO sources (title, url, publisher)
        VALUES (?, ?, ?)
    """, (title, url, publisher))

chemical_remedies = [
    {
        "disease": "Rice_Blast",
        "description": (
            "TNAU lists tricyclazole as a fungicide option for rice blast. "
            "Confirm the currently approved product and label directions "
            "before recommending or applying it."
        ),
        "description_mr": (
            "TNAU ने भातावरील ब्लास्ट रोगासाठी ट्रायसायक्लाझोल या "
            "बुरशीनाशकाचा पर्याय नमूद केला आहे. वापरण्यापूर्वी "
            "सध्याची मान्यता आणि उत्पादनाच्या लेबलवरील सूचना तपासा."
        ),
        "precautions": (
            "Follow current Indian label directions and local agricultural "
            "advice. Do not infer a dose from this database description."
        ),
        "source": "TNAU Rice Blast Chemical Management",
    },
    {
        "disease": "Wheat_stripe_rust",
        "description": (
            "TNAU lists mancozeb as a management option for wheat stripe rust. "
            "Verify current crop-specific registration and label instructions."
        ),
        "description_mr": (
            "TNAU ने गव्हावरील पिवळ्या तांबेरा रोगाच्या व्यवस्थापनासाठी "
            "मॅन्कोझेबचा पर्याय नमूद केला आहे. सध्याची मान्यता आणि "
            "लेबलवरील सूचना तपासा."
        ),
        "precautions": (
            "Use only if currently approved for this crop and disease; "
            "follow the product label and local agricultural advice."
        ),
        "source": "TNAU Wheat Stripe Rust Management",
    },
    {
        "disease": "Cotton_Aphids",
        "description": (
            "TNAU lists flonicamid 50% WG among insecticide options for cotton "
            "aphids when the economic threshold is reached. Verify current "
            "registration and label directions."
        ),
        "description_mr": (
            "कापसावरील माव्याचा प्रादुर्भाव आर्थिक नुकसान पातळीपर्यंत "
            "पोहोचल्यास TNAU ने फ्लोनिकामिड ५०% WG हा पर्याय नमूद केला आहे. "
            "सध्याची मान्यता आणि लेबलवरील सूचना तपासा."
        ),
        "precautions": (
            "Confirm the threshold and current label for cotton aphids. "
            "Protect pollinators and beneficial insects; follow PPE and "
            "pre-harvest interval instructions."
        ),
        "source": "TNAU Cotton Aphid Integrated Management",
    },
    {
        "disease": "Corn_Common_Rust",
        "description": (
            "TNAU's Crop Production Guide 2020 lists kresoxim-methyl 44.3% SC "
            "for maize rust. This is an older recommendation and requires "
            "current registration and label verification."
        ),
        "description_mr": (
            "TNAU च्या २०२० च्या पीक उत्पादन मार्गदर्शिकेत मक्यावरील "
            "तांबेरा रोगासाठी क्रेसॉक्सिम-मिथाइल ४४.३% SC नमूद केले आहे. "
            "ही जुनी शिफारस असल्याने सध्याची मान्यता तपासणे आवश्यक आहे."
        ),
        "precautions": (
            "Older guidance only. Do not use unless the current Indian "
            "product label permits use on maize for this disease."
        ),
        "source": "TNAU Crop Production Guide 2020 - Maize Rust",
    },
]

for item in chemical_remedies:
    disease_row = conn.execute(
        "SELECT id FROM diseases WHERE name = ?",
        (item["disease"],)
    ).fetchone()

    if disease_row is None:
        print("SKIPPED: disease not found:", item["disease"])
        continue

    disease_id = disease_row[0]

    existing = conn.execute("""
        SELECT id FROM remediations
        WHERE disease_id = ? AND type = 'chemical' AND description = ?
    """, (disease_id, item["description"])).fetchone()

    if existing:
        remediation_id = existing[0]
        print("Already exists:", item["disease"])
    else:
        cursor = conn.execute("""
            INSERT INTO remediations
                (disease_id, type, description, precautions,
                 description_mr, precautions_mr)
            VALUES (?, 'chemical', ?, ?, ?, ?)
        """, (
            disease_id,
            item["description"],
            item["precautions"],
            item["description_mr"],
            "वापरण्यापूर्वी स्थानिक कृषी तज्ज्ञांचा सल्ला घ्या "
            "आणि उत्पादनाच्या लेबलवरील सुरक्षिततेच्या सूचनांचे पालन करा."
        ))
        remediation_id = cursor.lastrowid

    source_id = conn.execute(
        "SELECT id FROM sources WHERE title = ?",
        (item["source"],)
    ).fetchone()[0]

    conn.execute("""
        INSERT OR IGNORE INTO remediation_sources
            (remediation_id, source_id)
        VALUES (?, ?)
    """, (remediation_id, source_id))

conn.commit()
print("Chemical remediation insertion completed.")


Already exists: Rice_Blast
Already exists: Wheat_stripe_rust
Chemical remediation insertion completed.


In [26]:
for item in chemical_remedies:
    result = conn.execute(
        "SELECT id FROM sources WHERE title = ?",
        (item["source"],)
    ).fetchone()

    if result is None:
        print("Missing:", item["source"])

Missing: TNAU Wheat Stripe Rust Management
Missing: TNAU Cotton Aphid Integrated Management


In [27]:
for title, url, publisher in chemical_sources:
    conn.execute("""
        INSERT OR IGNORE INTO sources (title, url, publisher)
        VALUES (?, ?, ?)
    """, (title, url, publisher))

conn.commit()

In [30]:
missing_sources = [
    (
        "TNAU Wheat Stripe Rust Management",
        "https://agritech.tnau.ac.in/crop_protection/crop_prot_crop%20diseases_cereals_wheat_3.html",
        "Tamil Nadu Agricultural University"
    ),
    (
        "TNAU Cotton Aphid Integrated Management",
        "https://agritech.tnau.ac.in/crop_protection/cotton/crop_prot_crop_insectpest%20_cotton_11.html",
        "Tamil Nadu Agricultural University"
    )
]

for title, url, publisher in missing_sources:
    conn.execute("""
        INSERT OR IGNORE INTO sources (title, url, publisher)
        VALUES (?, ?, ?)
    """, (title, url, publisher))

conn.commit()

for item in chemical_remedies:
    result = conn.execute(
        "SELECT id FROM sources WHERE title = ?",
        (item["source"],)
    ).fetchone()
    print(item["source"], "->", result)

TNAU Rice Blast Chemical Management -> (8,)
TNAU Wheat Stripe Rust Management -> None
TNAU Cotton Aphid Integrated Management -> None
TNAU Crop Production Guide 2020 - Maize Rust -> (11,)


In [32]:
conn.rollback()

In [36]:
for item in chemical_remedies:
    print(
        item["source"],
        "->",
        conn.execute(
            "SELECT id FROM sources WHERE title = ?",
            (item["source"],)
        ).fetchone()
    )

TNAU Rice Blast Chemical Management -> (8,)
TNAU Wheat Stripe Rust Management -> (6,)
TNAU Cotton Aphid Integrated Management -> (4,)
TNAU Crop Production Guide 2020 - Maize Rust -> (11,)


In [ ]:

chemical_sources = [
    (
        "TNAU Rice Blast Chemical Management",
        "https://agritech.tnau.ac.in/expert_system/paddy/cpdisblast.html",
        "Tamil Nadu Agricultural University"
    ),
    (
        "TNAU Wheat Stripe Rust Management",
        "https://agritech.tnau.ac.in/crop_protection/crop_prot_crop%20diseases_cereals_wheat_3.html",
        "Tamil Nadu Agricultural University"
    ),
    (
        "TNAU Cotton Aphid Integrated Management",
        "https://agritech.tnau.ac.in/crop_protection/cotton/crop_prot_crop_insectpest%20_cotton_11.html",
        "Tamil Nadu Agricultural University"
    ),
    (
        "TNAU Crop Production Guide 2020 - Maize Rust",
        "https://tnau.ac.in/site/research/wp-content/uploads/sites/60/2020/02/Agriculture-CPG-2020.pdf",
        "Tamil Nadu Agricultural University"
    ),
]

for title, url, publisher in chemical_sources:
    conn.execute("""
        INSERT OR IGNORE INTO sources (title, url, publisher)
        VALUES (?, ?, ?)
    """, (title, url, publisher))

chemical_remedies = [
    {
        "disease": "Rice_Blast",
        "description": (
            "TNAU lists tricyclazole as a fungicide option for rice blast. "
            "Confirm the currently approved product and label directions "
            "before recommending or applying it."
        ),
        "description_mr": (
            "TNAU ने भातावरील ब्लास्ट रोगासाठी ट्रायसायक्लाझोल या "
            "बुरशीनाशकाचा पर्याय नमूद केला आहे. वापरण्यापूर्वी "
            "सध्याची मान्यता आणि उत्पादनाच्या लेबलवरील सूचना तपासा."
        ),
        "precautions": (
            "Follow current Indian label directions and local agricultural "
            "advice. Do not infer a dose from this database description."
        ),
        "source": "TNAU Rice Blast Chemical Management",
    },
    {
        "disease": "Wheat_stripe_rust",
        "description": (
            "TNAU lists mancozeb as a management option for wheat stripe rust. "
            "Verify current crop-specific registration and label instructions."
        ),
        "description_mr": (
            "TNAU ने गव्हावरील पिवळ्या तांबेरा रोगाच्या व्यवस्थापनासाठी "
            "मॅन्कोझेबचा पर्याय नमूद केला आहे. सध्याची मान्यता आणि "
            "लेबलवरील सूचना तपासा."
        ),
        "precautions": (
            "Use only if currently approved for this crop and disease; "
            "follow the product label and local agricultural advice."
        ),
        "source": "TNAU Wheat Stripe Rust Management",
    },
    {
        "disease": "Cotton_Aphids",
        "description": (
            "TNAU lists flonicamid 50% WG among insecticide options for cotton "
            "aphids when the economic threshold is reached. Verify current "
            "registration and label directions."
        ),
        "description_mr": (
            "कापसावरील माव्याचा प्रादुर्भाव आर्थिक नुकसान पातळीपर्यंत "
            "पोहोचल्यास TNAU ने फ्लोनिकामिड ५०% WG हा पर्याय नमूद केला आहे. "
            "सध्याची मान्यता आणि लेबलवरील सूचना तपासा."
        ),
        "precautions": (
            "Confirm the threshold and current label for cotton aphids. "
            "Protect pollinators and beneficial insects; follow PPE and "
            "pre-harvest interval instructions."
        ),
        "source": "TNAU Cotton Aphid Integrated Management",
    },
    {
        "disease": "Corn_Common_Rust",
        "description": (
            "TNAU's Crop Production Guide 2020 lists kresoxim-methyl 44.3% SC "
            "for maize rust. This is an older recommendation and requires "
            "current registration and label verification."
        ),
        "description_mr": (
            "TNAU च्या २०२० च्या पीक उत्पादन मार्गदर्शिकेत मक्यावरील "
            "तांबेरा रोगासाठी क्रेसॉक्सिम-मिथाइल ४४.३% SC नमूद केले आहे. "
            "ही जुनी शिफारस असल्याने सध्याची मान्यता तपासणे आवश्यक आहे."
        ),
        "precautions": (
            "Older guidance only. Do not use unless the current Indian "
            "product label permits use on maize for this disease."
        ),
        "source": "TNAU Crop Production Guide 2020 - Maize Rust",
    },
]

for item in chemical_remedies:
    disease_row = conn.execute(
        "SELECT id FROM diseases WHERE name = ?",
        (item["disease"],)
    ).fetchone()

    if disease_row is None:
        print("SKIPPED: disease not found:", item["disease"])
        continue

    disease_id = disease_row[0]

    existing = conn.execute("""
        SELECT id FROM remediations
        WHERE disease_id = ? AND type = 'chemical' AND description = ?
    """, (disease_id, item["description"])).fetchone()

    if existing:
        remediation_id = existing[0]
        print("Already exists:", item["disease"])
    else:
        cursor = conn.execute("""
            INSERT INTO remediations
                (disease_id, type, description, precautions,
                 description_mr, precautions_mr)
            VALUES (?, 'chemical', ?, ?, ?, ?)
        """, (
            disease_id,
            item["description"],
            item["precautions"],
            item["description_mr"],
            "वापरण्यापूर्वी स्थानिक कृषी तज्ज्ञांचा सल्ला घ्या "
            "आणि उत्पादनाच्या लेबलवरील सुरक्षिततेच्या सूचनांचे पालन करा."
        ))
        remediation_id = cursor.lastrowid

    source_id = conn.execute(
        "SELECT id FROM sources WHERE title = ?",
        (item["source"],)
    ).fetchone()[0]

    conn.execute("""
        INSERT OR IGNORE INTO remediation_sources
            (remediation_id, source_id)
        VALUES (?, ?)
    """, (remediation_id, source_id))

conn.commit()
print("Chemical remediation insertion completed.")


Already exists: Rice_Blast
Already exists: Wheat_stripe_rust


TypeError: 'NoneType' object is not subscriptable